# 01 · Generate the datasets

Builds the raw material for the whole project from the public **Amazon Reviews 2023** corpus
(McAuley Lab, via Hugging Face).

1. Stream reviews from 10 product categories (min. 15 words), cap 5-star reviews at 40 % of the
   training pool, and draw a separate human-labeling **holdout** (no overlap with train, no duplicates).
2. Split the pool into train / test files.
3. Rare emotions (Fear, Guilt, Sadness) are scarce in organic reviews, so a cheap preliminary LLM screen
   finds extra candidates and tops up the holdout and train sets.

**Outputs** (written to `DATA_DIR`): `train_test_raw.csv`, `holdout_raw.csv`, `train_final.csv`,
`test_final.csv`, `buffer_remainder.csv` and the second-pass top-ups.
The holdout is labeled by humans outside this repo.

**Needs:** `OPENAI_API_KEY` (preliminary screen only, a few cents).
Uses `datasets==2.17.0` because the Amazon corpus ships a loading script (`trust_remote_code=True`).


In [ ]:
# ── Setup (runs locally or on Colab) ─────────────────────────────────────────
# Install dependencies first:  pip install -r requirements.txt
import os

# Where datasets, LLM labels, checkpoints and models are written (not committed to git).
# Colab: mount Drive and set CAPSTONE_DATA_DIR to a Drive folder to persist between sessions.
DATA_DIR = os.environ.get("CAPSTONE_DATA_DIR", "data")
os.makedirs(DATA_DIR, exist_ok=True)
OUTPUT_DIR = DATA_DIR

# API keys: environment variables first, Colab Secrets as a fallback.
_ENV_NAMES = {
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "xai": "XAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "minimax": "MINIMAX_API_KEY",
}

def get_api_key(name):
    value = os.environ.get(_ENV_NAMES.get(name, name)) or os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata as _colab_userdata
        return _colab_userdata.get(name)
    except Exception:
        return None

class userdata:  # drop-in for google.colab.userdata
    get = staticmethod(get_api_key)


In [ ]:
import random
import json
import pandas as pd
from datasets import load_dataset

TRAIN_SEED   = 42
HOLDOUT_SEED = 99

CATEGORIES = [
    "Baby_Products",
    "Pet_Supplies",
    "Beauty_and_Personal_Care",
    "Grocery_and_Gourmet_Food",
    "Toys_and_Games",
    "Health_and_Household",
    "Musical_Instruments",
    "Arts_Crafts_and_Sewing",
    "Home_and_Kitchen",
    "Sports_and_Outdoors",
]

# Collect a large buffer per category so the 5-star cap
# still leaves enough records to hit 20,000 total
TRAIN_BUFFER_PER_CAT = 8000
HOLDOUT_PER_CAT      = 130
MIN_WORDS            = 15

In [ ]:
def collect_from_category(category, n_train_buffer, n_holdout, train_seed, holdout_seed):
    print(f"\nLoading: {category}")

    ds = load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023",
        f"raw_review_{category}",
        split="full",
        streaming=True,
        trust_remote_code=True,
    )

    def is_valid(review):
        text = (review.get("text") or "").strip()
        return bool(text) and len(text.split()) >= MIN_WORDS

    def make_record(review, category):
        return {
            "category":          category,
            "rating":            review.get("rating"),
            "review_title":      (review.get("title") or "").strip(),
            "review_text":       (review.get("text") or "").strip(),
            "verified_purchase": review.get("verified_purchase"),
            "helpful_votes":     review.get("helpful_vote", 0),
            "user_id":           review.get("user_id", ""),
            "asin":              review.get("asin", ""),
            "timestamp":         review.get("timestamp"),
        }

    # Collect large train buffer, deduplicating as we go
    ds_train = ds.shuffle(seed=train_seed, buffer_size=50000)
    train_records  = []
    seen_texts = set()

    for review in ds_train:
        if len(train_records) >= n_train_buffer:
            break
        if not is_valid(review):
            continue
        key = review["text"].strip()[:120]
        if key in seen_texts:
            continue
        seen_texts.add(key)
        train_records.append(make_record(review, category))

    # Collect holdout with different seed, exclude all train texts
    ds_holdout = ds.shuffle(seed=holdout_seed, buffer_size=50000)
    holdout_records = []
    seen_holdout = set()

    for review in ds_holdout:
        if len(holdout_records) >= n_holdout:
            break
        if not is_valid(review):
            continue
        key = review["text"].strip()[:120]
        if key in seen_texts:       # not in train
            continue
        if key in seen_holdout:     # not already in holdout
            continue
        seen_holdout.add(key)
        holdout_records.append(make_record(review, category))

    print(f"  Buffer — Train: {len(train_records)} | Holdout: {len(holdout_records)}")
    return train_records, holdout_records

In [ ]:
all_train   = []
all_holdout = []

for cat in CATEGORIES:
    train_recs, holdout_recs = collect_from_category(
        cat,
        TRAIN_BUFFER_PER_CAT,
        HOLDOUT_PER_CAT,
        TRAIN_SEED,
        HOLDOUT_SEED,
    )
    all_train.extend(train_recs)
    all_holdout.extend(holdout_recs)

print(f"\nRaw buffer — Train: {len(all_train)} | Holdout: {len(all_holdout)}")


Loading: Baby_Products


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  Buffer — Train: 8000 | Holdout: 130

Loading: Pet_Supplies
  Buffer — Train: 8000 | Holdout: 130

Loading: Beauty_and_Personal_Care
  Buffer — Train: 8000 | Holdout: 130

Loading: Grocery_and_Gourmet_Food
  Buffer — Train: 8000 | Holdout: 130

Loading: Toys_and_Games
  Buffer — Train: 8000 | Holdout: 130

Loading: Health_and_Household
  Buffer — Train: 8000 | Holdout: 130

Loading: Musical_Instruments
  Buffer — Train: 8000 | Holdout: 130

Loading: Arts_Crafts_and_Sewing
  Buffer — Train: 8000 | Holdout: 130

Loading: Home_and_Kitchen
  Buffer — Train: 8000 | Holdout: 130

Loading: Sports_and_Outdoors
  Buffer — Train: 8000 | Holdout: 130

Raw buffer — Train: 80000 | Holdout: 1300


In [ ]:
df_train   = pd.DataFrame(all_train)
df_holdout = pd.DataFrame(all_holdout)

# --- Balance train: cap 5-star at 40% ---
# Strategy: keep ALL 1/2/3/4 star reviews, then sample just enough 5-star
non_five  = df_train[df_train["rating"] != 5.0].copy()
five_star = df_train[df_train["rating"] == 5.0].copy()

n_non_five  = len(non_five)
# If non-five is already more than 20k, trim it first
if n_non_five > 12000:
    non_five = non_five.sample(n=12000, random_state=TRAIN_SEED)
    n_non_five = 12000

# 5-star = 40% means non_five = 60%, so:
n_five_target = int(n_non_five * (40 / 60))
n_five_target = min(n_five_target, len(five_star), 8000)

five_sampled = five_star.sample(n=n_five_target, random_state=TRAIN_SEED)
df_train = pd.concat([non_five, five_sampled])

# Shuffle and trim to exactly 20,000
df_train = df_train.sample(frac=1, random_state=TRAIN_SEED).reset_index(drop=True)
df_train = df_train.iloc[:20000]

# --- Trim holdout to exactly 120 per category ---
df_holdout = (
    df_holdout
    .groupby("category", group_keys=False)
    .apply(lambda x: x.sample(n=min(120, len(x)), random_state=HOLDOUT_SEED))
    .reset_index(drop=True)
)

# --- Final checks ---
print(f"Final train size   : {len(df_train)}")
print(f"Final holdout size : {len(df_holdout)}")

print("\nRating distribution (train):")
rc = df_train["rating"].value_counts().sort_index()
for r, c in rc.items():
    pct = c / len(df_train) * 100
    bar = "#" * int(pct / 2)
    print(f"  {r} stars: {c:5d} ({pct:.1f}%) {bar}")

print("\nCategory distribution (train):")
print(df_train["category"].value_counts().to_string())

print("\nCategory distribution (holdout):")
print(df_holdout["category"].value_counts().to_string())

train_keys   = set(df_train["review_text"].str[:120])
holdout_keys = set(df_holdout["review_text"].str[:120])
print(f"\nOverlap train/holdout : {len(train_keys & holdout_keys)} (should be 0)")
print(f"Duplicates in train   : {df_train.duplicated(subset=['review_text']).sum()} (should be 0)")

Final train size   : 20000
Final holdout size : 1200

Rating distribution (train):
  1.0 stars:  1942 (9.7%) ####
  2.0 stars:  1484 (7.4%) ###
  3.0 stars:  2916 (14.6%) #######
  4.0 stars:  5658 (28.3%) ##############
  5.0 stars:  8000 (40.0%) ####################

Category distribution (train):
category
Beauty_and_Personal_Care    2087
Pet_Supplies                2069
Health_and_Household        2053
Grocery_and_Gourmet_Food    2047
Sports_and_Outdoors         2005
Musical_Instruments         2002
Baby_Products               1976
Home_and_Kitchen            1969
Arts_Crafts_and_Sewing      1918
Toys_and_Games              1874

Category distribution (holdout):
category
Arts_Crafts_and_Sewing      120
Baby_Products               120
Beauty_and_Personal_Care    120
Grocery_and_Gourmet_Food    120
Health_and_Household        120
Home_and_Kitchen            120
Musical_Instruments         120
Pet_Supplies                120
Sports_and_Outdoors         120
Toys_and_Games              1

/tmp/ipykernel_27228/2436001530.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(120, len(x)), random_state=HOLDOUT_SEED))


In [ ]:
df_train["word_count"]   = df_train["review_text"].str.split().str.len()
df_holdout["word_count"] = df_holdout["review_text"].str.split().str.len()

print("=== Word count stats (train) ===")
print(df_train["word_count"].describe().round(1).to_string())

print("\n=== Word count stats (holdout) ===")
print(df_holdout["word_count"].describe().round(1).to_string())

=== Word count stats (train) ===
count    20000.0
mean        83.5
std        100.4
min         15.0
25%         29.0
50%         53.0
75%         99.0
max       3497.0

=== Word count stats (holdout) ===
count    1200.0
mean       81.6
std       107.5
min        15.0
25%        27.0
50%        52.0
75%        92.0
max      1618.0


In [ ]:
import os
OUTPUT_DIR = DATA_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_train.to_json(
    f"{OUTPUT_DIR}/train_test_raw.json",
    orient="records", lines=True, force_ascii=False
)
df_holdout.to_json(
    f"{OUTPUT_DIR}/holdout_raw.json",
    orient="records", lines=True, force_ascii=False
)
df_train.to_csv(f"{OUTPUT_DIR}/train_test_raw.csv",   index=False)
df_holdout.to_csv(f"{OUTPUT_DIR}/holdout_raw.csv", index=False)

print(f"Saved train_test_raw  — {len(df_train)} records")
print(f"Saved holdout_raw     — {len(df_holdout)} records")

Saved train_test_raw  — 20000 records
Saved holdout_raw     — 1200 records


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import os

# Load the already-saved train file — do NOT re-collect anything
df = pd.read_csv(f"{DATA_DIR}/train_test_raw.csv")

# 80/20 split, stratified by category
train_split, test_split = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["category"]
)

train_split = train_split.reset_index(drop=True)
test_split  = test_split.reset_index(drop=True)

print(f"Train split : {len(train_split)} records")
print(f"Test split  : {len(test_split)} records")

print("\nRating distribution (train split):")
rc = train_split["rating"].value_counts().sort_index()
for r, c in rc.items():
    pct = c/len(train_split)*100
    print(f"  {r} stars: {c:5d} ({pct:.1f}%)")

print("\nCategory distribution (train split):")
print(train_split["category"].value_counts().to_string())

print("\nCategory distribution (test split):")
print(test_split["category"].value_counts().to_string())

# Save
OUTPUT_DIR = DATA_DIR
train_split.to_csv(f"{OUTPUT_DIR}/train_final.csv", index=False)
test_split.to_csv(f"{OUTPUT_DIR}/test_final.csv",   index=False)
print("\nSaved train_final.csv and test_final.csv")

Train split : 16000 records
Test split  : 4000 records

Rating distribution (train split):
  1.0 stars:  1580 (9.9%)
  2.0 stars:  1167 (7.3%)
  3.0 stars:  2346 (14.7%)
  4.0 stars:  4496 (28.1%)
  5.0 stars:  6411 (40.1%)

Category distribution (train split):
category
Beauty_and_Personal_Care    1670
Pet_Supplies                1655
Health_and_Household        1642
Grocery_and_Gourmet_Food    1638
Sports_and_Outdoors         1604
Musical_Instruments         1602
Baby_Products               1581
Home_and_Kitchen            1575
Arts_Crafts_and_Sewing      1534
Toys_and_Games              1499

Category distribution (test split):
category
Beauty_and_Personal_Care    417
Pet_Supplies                414
Health_and_Household        411
Grocery_and_Gourmet_Food    409
Sports_and_Outdoors         401
Musical_Instruments         400
Baby_Products               395
Home_and_Kitchen            394
Arts_Crafts_and_Sewing      384
Toys_and_Games              375

Saved train_final.csv and test_f

In [ ]:
import json
import time
import numpy as np
from tqdm import tqdm
from openai import OpenAI

In [ ]:
# Load existing datasets from Drive to rebuild deduplication keys
OUTPUT_DIR = DATA_DIR

df_train   = pd.read_csv(f"{OUTPUT_DIR}/train_test_raw.csv")
df_holdout = pd.read_csv(f"{OUTPUT_DIR}/holdout_raw.csv")

all_used_keys = (
    set(df_train["review_text"].str[:120]) |
    set(df_holdout["review_text"].str[:120])
)

print(f"Loaded train   : {len(df_train)}")
print(f"Loaded holdout : {len(df_holdout)}")
print(f"Total used keys: {len(all_used_keys)}")

# Collect remainder buffer — reviews not already in train or holdout
REMAINDER_SEED   = 42 + 99  # = 141, same deterministic seed as before
REMAINDER_TARGET = 3000     # per category

all_remainder = []

for cat in CATEGORIES:
    print(f"\nCollecting remainder: {cat}")
    ds = load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023",
        f"raw_review_{cat}",
        split="full",
        streaming=True,
        trust_remote_code=True,
    )
    ds_rem = ds.shuffle(seed=REMAINDER_SEED, buffer_size=50000)
    remainder_records = []
    seen_rem = set()

    for review in ds_rem:
        if len(remainder_records) >= REMAINDER_TARGET:
            break
        text = (review.get("text") or "").strip()
        if not text or len(text.split()) < 15:
            continue
        key = text[:120]
        if key in all_used_keys:
            continue
        if key in seen_rem:
            continue
        seen_rem.add(key)
        remainder_records.append({
            "category":          cat,
            "rating":            review.get("rating"),
            "review_title":      (review.get("title") or "").strip(),
            "review_text":       text,
            "verified_purchase": review.get("verified_purchase"),
            "helpful_votes":     review.get("helpful_vote", 0),
            "user_id":           review.get("user_id", ""),
            "asin":              review.get("asin", ""),
            "timestamp":         review.get("timestamp"),
        })
    all_remainder.extend(remainder_records)
    print(f"  Collected: {len(remainder_records)}")

df_remainder = pd.DataFrame(all_remainder)
df_remainder.to_csv(f"{OUTPUT_DIR}/buffer_remainder.csv", index=False)
print(f"\nTotal remainder: {len(df_remainder)}")

Loaded train   : 20000
Loaded holdout : 1200
Total used keys: 21197



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  Collected: 3000

  Collected: 3000

  Collected: 3000

  Collected: 3000

  Collected: 3000

  Collected: 3000

  Collected: 3000

  Collected: 3000

  Collected: 3000

  Collected: 3000

Total remainder: 30000


In [ ]:
import os
# ── Preliminary LLM labeling ─────────────────────────────────────────────────
EMOTIONS = [
    "Joy", "Love", "Surprise", "Pride", "Gratitude",
    "Contentment", "Relief", "Hope", "Interest",
    "Anger", "Frustration", "Disappointment",
    "Sadness", "Fear", "Guilt", "Disgust"
]

PRELIM_SYSTEM_PROMPT = """You are labeling consumer product reviews for a research project.

Read the review and identify which of the following 16 emotions are CLEARLY present.
Only label an emotion if there is direct, explicit evidence in the text.

Emotions:
- Joy: high-arousal excitement, delight, thrilled, ecstatic
- Love: long-term attachment, "my favorite", "can't live without", loyal
- Surprise: unexpected outcome, "didn't expect", "shocked", contrast from expectation
- Pride: self-credit for good decision, "I made the right choice", "smart buy"
- Gratitude: thankfulness toward product/company, "thank you", "so grateful"
- Contentment: calm satisfaction, "works fine", "no complaints", low intensity
- Relief: problem resolved, tension gone, "finally", "such a relief"
- Hope: future-oriented positive expectation, "hoping", "fingers crossed"
- Interest: curiosity, exploring features, "want to know more"
- Anger: blame, injustice, "furious", "unacceptable", "their fault"
- Frustration: blocked goals, repeated failure, "annoying", "keeps failing"
- Disappointment: unmet expectations, "let down", "fell short"
- Sadness: loss, emotional downturn, "feel sad", "really miss"
- Fear: threat, danger, risk, "worried", "scared", "unsafe"
- Guilt: self-blame, "my fault", "shouldn't have", personal regret
- Disgust: strong aversion, repulsion, "gross", "disgusting"

Rules:
1. Only label emotions with clear explicit evidence.
2. Multiple emotions can apply.
3. Return ONLY a JSON array of emotion strings from the list above.
4. If none clearly apply, return [].
5. No explanation, no other text.

Example output: ["Frustration", "Disappointment"]"""

ALLOWED_EMOTIONS = set(EMOTIONS)

def parse_emotion_labels(text):
    try:
        text = text.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return [e for e in parsed if e in ALLOWED_EMOTIONS]
    except:
        pass
    return []

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

def prelim_label_review(review_text, retries=3):
    wait_times = [3, 6, 15]
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": PRELIM_SYSTEM_PROMPT},
                    {"role": "user",   "content": review_text[:1500]}
                ],
                temperature=0,
                max_tokens=80,
            )
            return parse_emotion_labels(resp.choices[0].message.content)
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(wait_times[attempt])
            else:
                print(f"Error: {e}")
                return []

# Pre-filter by star rating to reduce API calls
# Negative rare emotions (Fear, Anger, Guilt, Disgust, Sadness) → low star
# Positive rare emotions (Pride, Hope) → high star
df_rem_low  = df_remainder[df_remainder["rating"] <= 2]
df_rem_mid  = df_remainder[df_remainder["rating"] == 3]
df_rem_high = df_remainder[df_remainder["rating"] >= 4]

df_to_label = pd.concat([
    df_rem_low.sample(min(1500, len(df_rem_low)),   random_state=42),
    df_rem_mid.sample(min(700,  len(df_rem_mid)),   random_state=42),
    df_rem_high.sample(min(800, len(df_rem_high)),  random_state=42),
]).drop_duplicates(subset=["review_text"]).reset_index(drop=True)

print(f"Reviews to label: {len(df_to_label)}")

# Run with checkpointing so you can resume if Colab disconnects
PRELIM_CHECKPOINT = f"{OUTPUT_DIR}/prelim_checkpoint.csv"

if os.path.exists(PRELIM_CHECKPOINT):
    df_done    = pd.read_csv(PRELIM_CHECKPOINT)
    done_keys  = set(df_done["review_text"].str[:120])
    df_to_run  = df_to_label[~df_to_label["review_text"].str[:120].isin(done_keys)]
    results    = df_done.to_dict("records")
    print(f"Resuming: {len(df_done)} done, {len(df_to_run)} remaining")
else:
    df_to_run = df_to_label.copy()
    results   = []
    print(f"Starting fresh: {len(df_to_run)} reviews")

for i, (_, row) in enumerate(tqdm(df_to_run.iterrows(), total=len(df_to_run))):
    emotions = prelim_label_review(row["review_text"])
    record   = row.to_dict()
    record["prelim_emotions"] = json.dumps(emotions)
    results.append(record)
    if (i + 1) % 100 == 0:
        pd.DataFrame(results).to_csv(PRELIM_CHECKPOINT, index=False)

df_prelim = pd.DataFrame(results)
df_prelim["prelim_emotions"] = df_prelim["prelim_emotions"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)
df_prelim.to_csv(f"{OUTPUT_DIR}/prelim_labels.csv", index=False)

print("\nEmotion distribution in remainder:")
for em in EMOTIONS:
    count = df_prelim["prelim_emotions"].apply(lambda x: em in x).sum()
    print(f"  {em}: {count}")

Reviews to label: 3000
Starting fresh: 3000 reviews


100%|██████████| 3000/3000 [27:44<00:00,  1.80it/s]


Emotion distribution in remainder:
  Joy: 432
  Love: 344
  Surprise: 73
  Pride: 104
  Gratitude: 82
  Contentment: 642
  Relief: 37
  Hope: 74
  Interest: 95
  Anger: 182
  Frustration: 1812
  Disappointment: 1971
  Sadness: 25
  Fear: 48
  Guilt: 23
  Disgust: 131


In [ ]:
from sklearn.model_selection import train_test_split
# ── Current holdout counts from human labeling ────────────────────────────────
CURRENT_HOLDOUT_COUNTS = {
    "Joy": 489, "Love": 276, "Surprise": 184, "Pride": 37,
    "Gratitude": 107, "Contentment": 925, "Relief": 142,
    "Hope": 60, "Interest": 207, "Anger": 59, "Frustration": 271,
    "Disappointment": 372, "Sadness": 48, "Fear": 22,
    "Guilt": 25, "Disgust": 37
}
MIN_PER_EMOTION = 80

# Keys already used — nothing from these can be added
existing_keys = (
    set(df_train["review_text"].str[:120]) |
    set(df_holdout["review_text"].str[:120])
)

# Pool = prelim-labeled reviews not already in any dataset
df_pool = df_prelim[
    ~df_prelim["review_text"].str[:120].isin(existing_keys)
].copy()
df_pool["num_emotions"] = df_pool["prelim_emotions"].apply(len)
print(f"Candidate pool: {len(df_pool)}")

# ── Holdout top-up: rarest emotion first (Ringel Cell 49 logic) ──────────────
np.random.seed(42)
gaps = {em: MIN_PER_EMOTION - cnt
        for em, cnt in CURRENT_HOLDOUT_COUNTS.items()
        if MIN_PER_EMOTION - cnt > 0}

emotions_by_rarity = sorted(gaps.keys(), key=lambda e: CURRENT_HOLDOUT_COUNTS[e])
emotion_counts     = dict(CURRENT_HOLDOUT_COUNTS)
holdout_selected   = set()

print("\nHoldout top-up:")
for rare_em in emotions_by_rarity:
    candidates = df_pool[
        (df_pool["prelim_emotions"].apply(lambda x: rare_em in x)) &
        (~df_pool.index.isin(holdout_selected))
    ].sort_values("num_emotions", ascending=False)

    for idx, row in candidates.iterrows():
        if emotion_counts[rare_em] >= MIN_PER_EMOTION:
            break
        holdout_selected.add(idx)
        for em in row["prelim_emotions"]:
            if em in emotion_counts:
                emotion_counts[em] += 1

    print(f"  {rare_em}: → {emotion_counts[rare_em]}")

df_holdout_topup = df_pool.loc[list(holdout_selected)].drop(
    columns=["prelim_emotions", "num_emotions"]
).reset_index(drop=True)

df_holdout_v2 = pd.concat([df_holdout, df_holdout_topup]).reset_index(drop=True)

# ── Train top-up: add rare-emotion reviews to train pool ─────────────────────
used_for_holdout = set(df_holdout_topup["review_text"].str[:120])
df_pool_train = df_pool[
    ~df_pool["review_text"].str[:120].isin(used_for_holdout)
].copy()

train_selected = set()
print("\nTrain top-up:")
for rare_em in emotions_by_rarity:
    candidates = df_pool_train[
        (df_pool_train["prelim_emotions"].apply(lambda x: rare_em in x)) &
        (~df_pool_train.index.isin(train_selected))
    ].sort_values("num_emotions", ascending=False).head(150)
    train_selected.update(candidates.index.tolist())
    print(f"  {rare_em}: adding {len(candidates)}")

df_train_topup = df_pool_train.loc[list(train_selected)].drop(
    columns=["prelim_emotions", "num_emotions"]
).reset_index(drop=True)

df_train_v2 = pd.concat([df_train, df_train_topup]).reset_index(drop=True)

# ── Re-split train/test ───────────────────────────────────────────────────────
train_v2, test_v2 = train_test_split(
    df_train_v2, test_size=0.2, random_state=42, stratify=df_train_v2["category"]
)
train_v2 = train_v2.reset_index(drop=True)
test_v2  = test_v2.reset_index(drop=True)

# ── Save ──────────────────────────────────────────────────────────────────────
df_holdout_v2.to_csv(f"{OUTPUT_DIR}/holdout_raw_v2.csv",   index=False)
df_train_v2.to_csv(f"{OUTPUT_DIR}/train_test_raw_v2.csv",  index=False)
train_v2.to_csv(f"{OUTPUT_DIR}/train_final_v2.csv",        index=False)
test_v2.to_csv(f"{OUTPUT_DIR}/test_final_v2.csv",          index=False)

# ── Final checks ──────────────────────────────────────────────────────────────
h_keys = set(df_holdout_v2["review_text"].str[:120])
t_keys = set(df_train_v2["review_text"].str[:120])
print(f"\nOverlap holdout/train : {len(h_keys & t_keys)} (should be 0)")
print(f"Holdout dups          : {df_holdout_v2.duplicated(subset=['review_text']).sum()} (should be 0)")
print(f"\nSummary:")
print(f"  holdout  : {len(df_holdout)} → {len(df_holdout_v2)}")
print(f"  train/test: {len(df_train)} → {len(df_train_v2)}")
print(f"  train split: {len(train_v2)}")
print(f"  test split : {len(test_v2)}")

Candidate pool: 3000

Holdout top-up:
  Fear: → 70
  Guilt: → 48
  Pride: → 80
  Disgust: → 80
  Sadness: → 73
  Anger: → 80
  Hope: → 80

Train top-up:
  Fear: adding 0
  Guilt: adding 0
  Pride: adding 61
  Disgust: adding 87
  Sadness: adding 0
  Anger: adding 150
  Hope: adding 54

Overlap holdout/train : 0 (should be 0)
Holdout dups          : 0 (should be 0)

Summary:
  holdout  : 1200 → 1400
  train/test: 20000 → 20352
  train split: 16281
  test split : 4071


In [ ]:
# ── Second pass: target Fear, Guilt, Sadness specifically ────────────────────
# Load v2 holdout and train to rebuild used keys
df_holdout_v2 = pd.read_csv(f"{OUTPUT_DIR}/holdout_raw_v2.csv")
df_train_v2   = pd.read_csv(f"{OUTPUT_DIR}/train_test_raw_v2.csv")

used_keys_v2 = (
    set(df_holdout_v2["review_text"].str[:120]) |
    set(df_train_v2["review_text"].str[:120])
)

# Also exclude what we already labeled in the first prelim pass
df_prelim_existing = pd.read_csv(f"{OUTPUT_DIR}/prelim_labels.csv")
already_labeled = set(df_prelim_existing["review_text"].str[:120])

all_used = used_keys_v2 | already_labeled

print(f"Total keys to exclude: {len(all_used)}")

# For Fear, Guilt, Sadness — these are almost exclusively in 1-2 star reviews
# Collect a bigger targeted batch of low-star reviews only
REMAINDER_SEED_2  = 999  # new seed for fresh reviews
REMAINDER_TARGET_2 = 5000  # per category, but we'll filter heavily by star rating

all_remainder_2 = []

for cat in CATEGORIES:
    print(f"\nCollecting: {cat}")
    ds = load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023",
        f"raw_review_{cat}",
        split="full",
        streaming=True,
        trust_remote_code=True,
    )
    ds_rem = ds.shuffle(seed=REMAINDER_SEED_2, buffer_size=50000)
    records = []
    seen    = set()

    for review in ds_rem:
        if len(records) >= REMAINDER_TARGET_2:
            break
        text   = (review.get("text") or "").strip()
        rating = review.get("rating")
        if not text or len(text.split()) < 15:
            continue
        # Only take 1-2 star reviews this time — much higher density of Fear/Guilt/Sadness
        if rating not in [1.0, 2.0]:
            continue
        key = text[:120]
        if key in all_used:
            continue
        if key in seen:
            continue
        seen.add(key)
        records.append({
            "category":          cat,
            "rating":            rating,
            "review_title":      (review.get("title") or "").strip(),
            "review_text":       text,
            "verified_purchase": review.get("verified_purchase"),
            "helpful_votes":     review.get("helpful_vote", 0),
            "user_id":           review.get("user_id", ""),
            "asin":              review.get("asin", ""),
            "timestamp":         review.get("timestamp"),
        })
    all_remainder_2.extend(records)
    print(f"  Collected: {len(records)}")

df_remainder_2 = pd.DataFrame(all_remainder_2)
print(f"\nTotal new remainder: {len(df_remainder_2)}")

# Label with LLM — cap at 2000 to keep cost low
df_to_label_2 = df_remainder_2.sample(
    min(2000, len(df_remainder_2)), random_state=42
).reset_index(drop=True)

print(f"Sending to LLM: {len(df_to_label_2)}")

PRELIM_CHECKPOINT_2 = f"{OUTPUT_DIR}/prelim_checkpoint_2.csv"

if os.path.exists(PRELIM_CHECKPOINT_2):
    df_done   = pd.read_csv(PRELIM_CHECKPOINT_2)
    done_keys = set(df_done["review_text"].str[:120])
    df_to_run = df_to_label_2[~df_to_label_2["review_text"].str[:120].isin(done_keys)]
    results   = df_done.to_dict("records")
    print(f"Resuming: {len(df_done)} done, {len(df_to_run)} remaining")
else:
    df_to_run = df_to_label_2.copy()
    results   = []
    print(f"Starting fresh: {len(df_to_run)} reviews")

for i, (_, row) in enumerate(tqdm(df_to_run.iterrows(), total=len(df_to_run))):
    emotions = prelim_label_review(row["review_text"])
    record   = row.to_dict()
    record["prelim_emotions"] = json.dumps(emotions)
    results.append(record)
    if (i + 1) % 100 == 0:
        pd.DataFrame(results).to_csv(PRELIM_CHECKPOINT_2, index=False)

df_prelim_2 = pd.DataFrame(results)
df_prelim_2["prelim_emotions"] = df_prelim_2["prelim_emotions"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)
df_prelim_2.to_csv(f"{OUTPUT_DIR}/prelim_labels_2.csv", index=False)

print("\nEmotion counts in new batch:")
for em in ["Fear", "Guilt", "Sadness"]:
    count = df_prelim_2["prelim_emotions"].apply(lambda x: em in x).sum()
    print(f"  {em}: {count}")

Total keys to exclude: 24197

Collecting: Baby_Products
  Collected: 5000

Collecting: Pet_Supplies
  Collected: 5000

Collecting: Beauty_and_Personal_Care
  Collected: 5000

Collecting: Grocery_and_Gourmet_Food
  Collected: 5000

Collecting: Toys_and_Games
  Collected: 5000

Collecting: Health_and_Household
  Collected: 5000

Collecting: Musical_Instruments
  Collected: 5000

Collecting: Arts_Crafts_and_Sewing
  Collected: 5000

Collecting: Home_and_Kitchen
  Collected: 5000

Collecting: Sports_and_Outdoors
  Collected: 5000

Total new remainder: 50000
Sending to LLM: 2000
Starting fresh: 2000 reviews


100%|██████████| 2000/2000 [18:08<00:00,  1.84it/s]


Emotion counts in new batch:
  Fear: 40
  Guilt: 25
  Sadness: 36


In [ ]:
# ── Updated current counts after v2 ──────────────────────────────────────────
CURRENT_COUNTS_V2 = {
    "Joy": 489, "Love": 276, "Surprise": 184, "Pride": 80,
    "Gratitude": 107, "Contentment": 925, "Relief": 142,
    "Hope": 80, "Interest": 207, "Anger": 80, "Frustration": 271,
    "Disappointment": 372, "Sadness": 73, "Fear": 70,
    "Guilt": 48, "Disgust": 80
}

# Only targeting the three still under 80
still_needed = {em: 80 - cnt
                for em, cnt in CURRENT_COUNTS_V2.items()
                if 80 - cnt > 0}
print("Still needed:", still_needed)

# Pool = new prelim batch, excluding everything already used
used_keys_v2 = (
    set(df_holdout_v2["review_text"].str[:120]) |
    set(df_train_v2["review_text"].str[:120])
)
df_pool_2 = df_prelim_2[
    ~df_prelim_2["review_text"].str[:120].isin(used_keys_v2)
].copy()
df_pool_2["num_emotions"] = df_pool_2["prelim_emotions"].apply(len)
print(f"Candidate pool: {len(df_pool_2)}")

# ── Holdout top-up ────────────────────────────────────────────────────────────
emotion_counts_2  = dict(CURRENT_COUNTS_V2)
holdout_selected_2 = set()
emotions_by_rarity_2 = sorted(still_needed.keys(), key=lambda e: CURRENT_COUNTS_V2[e])

print("\nHoldout top-up (second pass):")
for rare_em in emotions_by_rarity_2:
    candidates = df_pool_2[
        (df_pool_2["prelim_emotions"].apply(lambda x: rare_em in x)) &
        (~df_pool_2.index.isin(holdout_selected_2))
    ].sort_values("num_emotions", ascending=False)

    for idx, row in candidates.iterrows():
        if emotion_counts_2[rare_em] >= 80:
            break
        holdout_selected_2.add(idx)
        for em in row["prelim_emotions"]:
            if em in emotion_counts_2:
                emotion_counts_2[em] += 1

    print(f"  {rare_em}: → {emotion_counts_2[rare_em]}")

df_holdout_topup_2 = df_pool_2.loc[list(holdout_selected_2)].drop(
    columns=["prelim_emotions", "num_emotions"]
).reset_index(drop=True)

df_holdout_v3 = pd.concat([df_holdout_v2, df_holdout_topup_2]).reset_index(drop=True)

# ── Train top-up from leftovers ───────────────────────────────────────────────
used_for_holdout_2 = set(df_holdout_topup_2["review_text"].str[:120])
df_pool_train_2 = df_pool_2[
    ~df_pool_2["review_text"].str[:120].isin(used_for_holdout_2)
].copy()

train_selected_2 = set()
print("\nTrain top-up (second pass):")
for rare_em in emotions_by_rarity_2:
    candidates = df_pool_train_2[
        (df_pool_train_2["prelim_emotions"].apply(lambda x: rare_em in x)) &
        (~df_pool_train_2.index.isin(train_selected_2))
    ].sort_values("num_emotions", ascending=False).head(150)
    train_selected_2.update(candidates.index.tolist())
    print(f"  {rare_em}: adding {len(candidates)}")

df_train_topup_2 = df_pool_train_2.loc[list(train_selected_2)].drop(
    columns=["prelim_emotions", "num_emotions"]
).reset_index(drop=True)

df_train_v3 = pd.concat([df_train_v2, df_train_topup_2]).reset_index(drop=True)

# ── Re-split ──────────────────────────────────────────────────────────────────
train_v3, test_v3 = train_test_split(
    df_train_v3, test_size=0.2, random_state=42, stratify=df_train_v3["category"]
)
train_v3 = train_v3.reset_index(drop=True)
test_v3  = test_v3.reset_index(drop=True)

# ── Save ──────────────────────────────────────────────────────────────────────
df_holdout_v3.to_csv(f"{OUTPUT_DIR}/holdout_raw_v3.csv",   index=False)
df_train_v3.to_csv(f"{OUTPUT_DIR}/train_test_raw_v3.csv",  index=False)
train_v3.to_csv(f"{OUTPUT_DIR}/train_final_v3.csv",        index=False)
test_v3.to_csv(f"{OUTPUT_DIR}/test_final_v3.csv",          index=False)

# ── Checks ────────────────────────────────────────────────────────────────────
h_keys = set(df_holdout_v3["review_text"].str[:120])
t_keys = set(df_train_v3["review_text"].str[:120])
print(f"\nOverlap holdout/train : {len(h_keys & t_keys)} (should be 0)")
print(f"Holdout dups          : {df_holdout_v3.duplicated(subset=['review_text']).sum()} (should be 0)")
print(f"\nSummary:")
print(f"  holdout   : {len(df_holdout_v2)} → {len(df_holdout_v3)}")
print(f"  train/test: {len(df_train_v2)} → {len(df_train_v3)}")
print(f"  train split: {len(train_v3)}")
print(f"  test split : {len(test_v3)}")

Still needed: {'Sadness': 7, 'Fear': 10, 'Guilt': 32}
Candidate pool: 2000

Holdout top-up (second pass):
  Guilt: → 73
  Fear: → 80
  Sadness: → 80

Train top-up (second pass):
  Guilt: adding 0
  Fear: adding 30
  Sadness: adding 29

Overlap holdout/train : 0 (should be 0)
Holdout dups          : 0 (should be 0)

Summary:
  holdout   : 1400 → 1441
  train/test: 20352 → 20411
  train split: 16328
  test split : 4083
